In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from scipy.sparse import load_npz
import joblib
import pandas as pd
import numpy as np


data = joblib.load('/content/drive/MyDrive/Colab Notebooks/data project 1 (Sentiment Analysis) AI Indonesia/Individual Project/TF-IDF&LogisticRegression/data/alldata_tfidf_optimal.pkl')
x_train = data['x_train_tfidf_optimal']
x_test = data['x_test_tfidf_optimal']
y_train = data['y_train_enc']
y_test = data['y_test_enc']
tfidf = data['tfidf_optimal']
encoder = data['encoder']

In [ ]:
# ============================================
# ENSEMBLE SEDERHANA (TIDAK KOMPLEKS)
# ============================================

from sklearn.ensemble import VotingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

print("\n" + "="*60)
print("📌 ENSEMBLE SEDERHANA")
print("="*60)

# 1. Hanya 2 model terbaik
best_models = [
    ('nb', MultinomialNB(alpha=0.5)),
    ('lr', LogisticRegression(C=0.1, max_iter=1000))
]

# Voting sederhana
voting_simple = VotingClassifier(estimators=best_models, voting='soft')
voting_simple.fit(x_train, y_train)
acc_simple = accuracy_score(y_test, voting_simple.predict(x_test))
print(f"🎯 Voting (NB + LR): {acc_simple:.4f}")

# 2. Weighted Voting (beri bobot berdasarkan performa)
weights = [0.6, 0.4]  # NB lebih besar karena 62.8%
voting_weighted = VotingClassifier(estimators=best_models, voting='soft', weights=weights)
voting_weighted.fit(x_train, y_train)
acc_weighted = accuracy_score(y_test, voting_weighted.predict(x_test))
print(f"🎯 Weighted Voting: {acc_weighted:.4f}")


📌 ENSEMBLE SEDERHANA
🎯 Voting (NB + LR): 0.6198
🎯 Weighted Voting: 0.6171


In [ ]:
# ============================================
# GRID SEARCH UNTUK VOTING
# ============================================

from sklearn.model_selection import GridSearchCV

# Cari bobot terbaik untuk voting
param_grid = {
    'weights': [[0.7, 0.3], [0.6, 0.4], [0.5, 0.5], [0.4, 0.6], [0.3, 0.7]]
}

grid_voting = GridSearchCV(
    VotingClassifier(estimators=best_models, voting='soft'),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid_voting.fit(x_train, y_train)
print(f"✅ Bobot terbaik: {grid_voting.best_params_}")
print(f"✅ Akurasi: {grid_voting.best_score_:.4f}")

✅ Bobot terbaik: {'weights': [0.3, 0.7]}
✅ Akurasi: 0.6075


In [ ]:
# 2. Weighted Voting (beri bobot berdasarkan performa)
weights = [0.3, 0.7]  # NB lebih besar karena 62.8%
voting_weighted = VotingClassifier(estimators=best_models, voting='soft', weights=weights)
voting_weighted.fit(x_train, y_train)
acc_weighted = accuracy_score(y_test, voting_weighted.predict(x_test))
print(f"🎯 Weighted Voting: {acc_weighted:.4f}")

🎯 Weighted Voting: 0.6281


In [7]:
# ============================================
# REKOMENDASI MODEL FINAL
# ============================================

print("\n" + "="*60)
print("🎯 REKOMENDASI MODEL FINAL")
print("="*60)

# Model 1: Naive Bayes (terbukti stabil)
final_nb = MultinomialNB(alpha=0.5)
final_nb.fit(x_train, y_train)

# Model 2: Voting NB + LR
final_voting = VotingClassifier([
    ('nb', MultinomialNB(alpha=0.5)),
    ('lr', LogisticRegression(C=0.1, max_iter=1000))
], voting='soft', weights=[0.6, 0.4])
final_voting.fit(x_train, y_train)

# Evaluasi
acc_final_nb = accuracy_score(y_test, final_nb.predict(x_test))
acc_final_voting = accuracy_score(y_test, final_voting.predict(x_test))

print(f"""
📊 Perbandingan Final:
   • Naive Bayes        : {acc_final_nb:.4f} ({acc_final_nb*100:.2f}%)
   • Voting NB + LR     : {acc_final_voting:.4f} ({acc_final_voting*100:.2f}%)

🏆 KESIMPULAN:
   Naive Bayes tetap menjadi model terbaik dengan 62.8%
   Ensemble tidak membantu karena dataset terlalu kecil/sparse
""")


🎯 REKOMENDASI MODEL FINAL

📊 Perbandingan Final:
   • Naive Bayes        : 0.6253 (62.53%)
   • Voting NB + LR     : 0.6171 (61.71%)

🏆 KESIMPULAN:
   Naive Bayes tetap menjadi model terbaik dengan 62.8%
   Ensemble tidak membantu karena dataset terlalu kecil/sparse

